# Does DFC add the cls sigmoid, or must we?

**The question.** Hailo's own `yolov11s.alls` carries three `change_output_activation(<cls_conv>, sigmoid)`
lines. The ALLS this repo generates carries none, and the YOLOv8 NMS op in HailoRT 4.20.0 applies no
sigmoid of its own — it compares the dequantized class value straight against `nms_scores_th`.

So either DFC inserts it for us, or **every HEF this repo has produced has been feeding raw logits
into a threshold that expects probabilities.**

**What this costs.** No device, no full compile, no calibration set. It parses the ONNX, applies the
*same* model script the pipeline uses, runs `optimize()` on a handful of frames — the stage where
`change_output_activation` would take effect — and prints what the output layers actually carry.
Quantization quality is irrelevant, because this is a question about the graph.

**Runtime.** ~10 min the first time (installing DFC), ~2 min after.

Then paste the whole output back. Loom Oracle (AI).


## 1 — Point at your files

You need two things in Drive. If you ran `compile_run.ipynb` before, they are already there:

| File | Where |
|---|---|
| `hailo_dataflow_compiler-3.*.whl` | `MyDrive/hailo/` |
| `best.onnx` (any run's export) | `MyDrive/hailo/` |

If the wheel is missing, download it from the Hailo Developer Zone (it is login-gated) and drop it there.


In [ ]:
HAILO_DIR = '/content/drive/MyDrive/hailo'   # wheel + onnx live here
ONNX_NAME = 'best.onnx'                      # any detection export from this repo
CALIB_DIR = None                             # optional: a folder of real frames. None = synthetic

REPO_URL  = 'https://github.com/pitikorn-pam/sack-train-ml.git'
BRANCH    = 'main'


## 2 — Mount Drive and check the two files are there


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import glob, os
wheels = glob.glob(f'{HAILO_DIR}/hailo_dataflow_compiler*.whl')
onnx_path = f'{HAILO_DIR}/{ONNX_NAME}'
print('DFC wheel :', wheels or 'NOT FOUND')
print('onnx      :', onnx_path, os.path.exists(onnx_path))
assert wheels, f'put the DFC wheel in {HAILO_DIR}'
assert os.path.exists(onnx_path), f'put {ONNX_NAME} in {HAILO_DIR}'
DFC_WHL = wheels[0]
print('\nready')


## 3 — Build the DFC virtualenv

DFC pins numpy/scipy versions that fight with Colab's own, so it lives in its own venv and is only
ever called as a subprocess. Slow cell — a few minutes.


In [ ]:
!pip -q install virtualenv
!virtualenv -q -p python3 /content/hailo_venv
VENV = '/content/hailo_venv/bin'
!{VENV}/pip -q install --upgrade pip setuptools wheel
!{VENV}/pip -q install numpy==1.23.3 scipy==1.10.1 pillow onnx
!{VENV}/pip install {DFC_WHL}
!{VENV}/python -c "import hailo_sdk_client as c; print('hailo_sdk_client', getattr(c,'__version__','?'))"


## 4 — Get the repo

The probe imports the real recipe from `scripts/compile_clientrunner.py` rather than restating it,
so the answer describes the pipeline that actually runs — not a copy of it that has drifted.


In [ ]:
import subprocess, pathlib
REPO = pathlib.Path('/content/sack-train-ml')
if REPO.exists():
    subprocess.run(['git','fetch','origin'], cwd=REPO, check=True)
    subprocess.run(['git','reset','--hard',f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO)], check=True)
print('repo at', subprocess.check_output(['git','rev-parse','--short','HEAD'], cwd=REPO, text=True).strip())


## 5 — Run the probe

Reads the output layers twice: straight after parsing, and after `optimize()` — the stage where a
`change_output_activation` would take effect. The layers whose channel count equals your class count
are the classification heads; the 64-channel ones are the DFL box heads.


In [ ]:
CALIB_ARG = f'--calib {CALIB_DIR}' if CALIB_DIR else ''
!{VENV}/python /content/sack-train-ml/scripts/probe_cls_activation.py \
    --onnx {onnx_path} --work /content/probe {CALIB_ARG}


## 6 — What DFC applied to itself, verbatim

A second, independent reading of the same question: the model script DFC generated for itself.
If `change_output_activation` appears here, DFC inserted it.


In [ ]:
!{VENV}/hailo har extract /content/probe/probe_probe.har \
    --auto-model-script-path /content/probe/effective.alls
print(open('/content/probe/effective.alls').read())


## 7 — Reading the result

| cls heads show | Means | What happens next |
|---|---|---|
| `sigmoid` | DFC inserts it | Hypothesis closed. The INT8 investigation stays on `optimization_level`. |
| `linear` / none | We are missing it | Our ALLS must emit it, and every HEF built here so far compared logits against a probability threshold. |

**Paste the whole output of cells 5 and 6 back — raw.** The printed lines are the evidence; the
verdict line at the bottom is only a convenience.
